### [1934. Confirmation Rate](https://leetcode.com/problems/confirmation-rate/)
**Difficulty:** Medium  
**Topic:** Left Join / Aggregate Functions

In [ ]:
# Write your MySQL query statement below
-- cr : confirmed / requests
-- round 2 decimals
with tbl as (
select s.user_id
    , coalesce(count(c.user_id),0) as total_request
    , coalesce(count(case when action = "confirmed" then c.user_id end),0) as confirmed_msg
from Signups s
left join Confirmations c on s.user_id = c.user_id
group by 1
)
select user_id, case when total_request = 0 then 0
                else round(confirmed_msg / total_request,2) end
                    as confirmation_rate
from tbl

### [1193. Monthly Transactions I](https://leetcode.com/problems/monthly-transactions-i/description/?envType=study-plan-v2&envId=top-sql-50)
**Difficulty:** Medium

In [ ]:
# Write your MySQL query statement below
/* monthly per country:
    -- number of trans
    --total amt
    -- approved trans
    */

select DATE_FORMAT(trans_date, '%Y-%m') AS month,
    , country
    , count(id) as trans_count
    , count(case when state = 'approved' then id end) as approved_count
    , sum(amount) as trans_total_amount
    , sum(case when state = 'approved' then amount else 0 end) as approved_total_amount
from Transactions
group by 1,2

### [1174. Immediate Food Delivery II](https://leetcode.com/problems/immediate-food-delivery-ii/description/?envType=study-plan-v2&envId=top-sql-50)
**Difficulty:** Medium

In [ ]:
# Write your MySQL query statement below
/*
-- imediate = customer_pref_delivery_date = order_date
conditions :
- for first order calcualte pct of imediate
- round 2 decimals

-- calc first order per users
-- calc imediate orders
-- calc pct

*/

with ranked_orders as (
    select *
    , row_number() over(partition by  customer_id order by  order_date) as rn
    from Delivery
)
select
round(100.00 * count( case when order_date = customer_pref_delivery_date then customer_id end )
    / count(customer_id),2)  as immediate_percentage
from ranked_orders
where rn = 1


##### more optimized solution


In [ ]:
select round(
    sum(case when order_date = customer_pref_delivery_date then 1 else 0 end) *100.00 / count(*)
    ,2) as immediate_percentage
from Delivery
where (customer_id,order_date) in (

    select customer_id, min(order_date)
    from Delivery
    group by 1
)


### [550. Game Play Analysis IV](https://leetcode.com/problems/game-play-analysis-iv/description/?envType=study-plan-v2&envId=top-sql-50)
**Difficulty:** Medium

**Topic:** Aggregate Functions

In [ ]:
# Write your MySQL query statement below
/*
pct of users logged day after first logged
codition:
-- numinator : # customers who logged day after
--denominator : # totoal players
-- round 2 decimals
*/
with active_players as (
select count(distinct player_id) as active_player
from Activity
where (player_id, event_date) in (
    select  player_id, date_add(min(event_date), INTERVAL 1 DAY )
    from Activity
    group by 1
))
select round((active_player
 / (select count(distinct player_id) from Activity )),2) as fraction
 from active_players


### [070. Product Sales Analysis III](https://leetcode.com/problems/product-sales-analysis-iii/description/?envType=study-plan-v2&envId=top-sql-50)
**Difficulty:** Medium

**Topic:** Aggregate Functions

In [ ]:
# Write your MySQL query statement below

-- find first year
-- select rows with prev condition

select product_id , year as first_year , quantity, price
from Sales
where (product_id, year) in (
    select product_id , min(year)
    from Sales
    group by 1
)

### [1045. Customers Who Bought All Products](https://leetcode.com/problems/customers-who-bought-all-products/description/?envType=study-plan-v2&envId=top-sql-50)
**Difficulty:** Medium
**Topic:** Aggregate Functions

In [ ]:
# Write your MySQL query statement below
-- find all uniq product id
-- find customers with condition that number of items buy equeal prev step


select customer_id
from Customer
group by 1
having count(distinct product_key)   = (select count(*) from Product)

### [180. Consecutive Numbers](https://leetcode.com/problems/consecutive-numbers/description/?envType=study-plan-v2&envId=top-sql-50)
**Difficulty:** Medium


In [ ]:
# Write your MySQL query statement below

with diffs as (
select num
    , lead(num) over(order by id) as next_num
    , lead(num,2) over(order by id) as next2_num

from Logs )
select  distinct num  as ConsecutiveNums
from diffs
where num = next_num and  num = next2_num


#### more optimized solution

In [ ]:
select distinct l1.num as ConsecutiveNums
from logs l1
join logs l2 on l1.id=l2.id-1
join logs l3 on l1.id=l3.id-2
where l1.num=l2.num and l2.num=l3.num;


### [1164. Product Price at a Given Date](https://leetcode.com/problems/product-price-at-a-given-date/description/?envType=study-plan-v2&envId=top-sql-50)
**Difficulty:** Medium


In [ ]:
# Write your MySQL query statement below

-- get latestest price before 2019-08-16.
-- if a product not in prev step return 10
with latest_changes as (
select product_id
    , new_price as price
    ,row_number() over(partition by product_id order by change_date desc) rn
from Products
where change_date <= '2019-08-16'

)
select distinct p1.product_id , COALESCE(p2.price ,10)  as price
from Products p1
left join latest_changes p2 on p1.product_id = p2.product_id and p2.rn=1

### [1204. Last Person to Fit in the Bus](https://leetcode.com/problems/last-person-to-fit-in-the-bus/description/?envType=study-plan-v2&envId=top-sql-50)
**Difficulty:** Medium

**Topic:** Advanced Select and Joins

In [ ]:
# Write your MySQL query statement below
with running_totalas as (
select person_name,sum(weight) over(order by turn) as running_total
from Queue
)
select person_name
from running_totalas
where running_total <=1000
order by running_total desc
limit 1

### [1907. Count Salary Categories](https://leetcode.com/problems/count-salary-categories/description/?envType=study-plan-v2&envId=top-sql-50)
**Difficulty:** Medium
**Topic:** Advanced Select and Joins


In [ ]:
# Write your MySQL query statement below
-- cateorze salry based on the bound range
-- count based on this range
with cats as (
    select 'Low Salary' as category
    union all
    select 'Average Salary' as category
    union all
    select 'High Salary' as category

)
, account_labels as (
    select *,
    case when income < 20000 then 'Low Salary'
    when income <= 50000 then 'Average Salary'
    when income > 50000 then 'High Salary' end as category
    from Accounts
)
select c.category , coalesce(count(a.account_id),0) as accounts_count
from cats c
left join account_labels a on c.category = a.category
group by 1

### [1341. Movie Rating](https://leetcode.com/problems/movie-rating/description/?envType=study-plan-v2&envId=top-sql-50)
**Difficulty:** Medium
**Topic:** Advanced Select and Joins

In [ ]:
# Write your MySQL query statement below
-- part1 :
    -- find the user with highest number of rating
    -- order by asc
    -- limit 1
-- part 2:
-- find movie with highest average rating in February 2020
 -- order by asc
 -- limit 1

(select u.name as results
 from MovieRating m
 join Users u on m.user_id = u.user_id
 group by 1
 order by count(*) desc , u.name
 limit 1)

 union all


(select m.title  as results
 from MovieRating mr
 join Movies m on mr.movie_id = m.movie_id
 where date_format(created_at, '%Y-%m') = '2020-02'
 group by 1
 order by avg(rating) desc , m.title
 limit 1);





### [1321. Restaurant Growth](https://leetcode.com/problems/restaurant-growth/description/?envType=study-plan-v2&envId=top-sql-50)
**Difficulty:** Medium
**Topic:** Advanced Select and Joins


In [ ]:
# Write your MySQL query statement below
-- moving avg last 6 day and current day
with daily_aggs as (
    select visited_on , sum(amount) as amount
    from Customer
    group by 1
)
,calcs as (
select visited_on
    , sum(amount) over(order by visited_on rows between 6 preceding and current row) as amount
    ,round(avg(amount) over(order by visited_on rows between 6 preceding and current row),2) as average_amount
    ,count(visited_on) over(order by visited_on rows between 6 preceding and current row) as cnt
from daily_aggs
group by 1
)
select visited_on, amount , average_amount
from calcs
where cnt = 7

### [602. Friend Requests II: Who Has the Most Friends](https://leetcode.com/problems/friend-requests-ii-who-has-the-most-friends/description/?envType=study-plan-v2&envId=top-sql-50)

**Difficulty:** Medium
**Topic:** Advanced Select and Joins



In [ ]:
# Write your MySQL query statement below
with tbl as (
select requester_id as id
from RequestAccepted
union all
select accepter_id as id
from RequestAccepted )
select id , count(*) as num
from tbl
group by 1
order by count(*) desc
limit 1


### [176. Second Highest Salary](https://leetcode.com/problems/second-highest-salary/description/?envType=study-plan-v2&envId=top-sql-50)
**Difficulty:** Medium

** Topic:** Advanced String Functions / Regex / Clause




In [ ]:
# Write your MySQL query statement below

with salary_uq as(
    select distinct salary
    from Employee
)
,ranked as(
select  salary , row_number() over(order by salary desc) as rn
from salary_uq
)
select case when max(rn) = 1 then null else coalesce(salary,null) end as SecondHighestSalary
from ranked
where rn =2


Write a query to find the top 3 highest-earning employees within each department. If multiple employees have the exact same salary, they must share the same rank, and the subsequent ranking numbers must not contain any gaps.

Expected Output Columns:
department_name, emp_name, salary

In [ ]:

with ranked as(
    select department_id
    ,emp_name
    ,salary
    ,dense_rank() over(partition by department_id order by salary desc) as rn
    from employees e
)
select d.department_name , emp_name, salary
from ranked
join departments d on ranked.department_id = d.department_id
where rn <= 3


In [ ]:
with daily_sales as (
select sale_date , sum(amount) as daily_total
from sales
GROUP by 1 )
SELECT sale_date, daily_total , avg(daily_total) over(order by sale_date Range between interval 2 day prededing and current row) as 3_day_moving_avg
from daily_sales


In [ ]:
-- find the first order
-- check it it was not the first order return the diff
with first_orders as (

SELECT customer_id, min(order_timestamp) as first_order
FROM orders
GROUP by 1
)

SELECT customer_id
    ,order_timestamp
    , case when order_timestamp = first_order then NULL else
    datediff(order_timestamp, LAG(order_timestamp,1) over( partition by customer_id order by order_timestamp))
    end as days_since_last_order
FROM orders o
join first_orders fo on o.customer_id = fo.customer_id

In [ ]:
with self_join as (
SELECT e.emp_name as employee_name
, e.salary as employee_salary
, m.emp_name as manager_name
, m.salary as manager_salary
FROM employees e
JOIN employees m on e.manager_id = m.emp_id
)
SELECT  *
from self_join
where employee_salary > manager_salary



In [ ]:
-- aproach 1 : with CTE and max
with max_shipment as (
    SELECT shipment_id , max(log_timestamp) as latest_time
    FROM shipment_logs
    GROUP by 1
)
SELECT sl.shipment_id
,sl.status_text as  current_status
,sl.log_timestamp as  last_updated
FROM shipment_logs sl
JOIN max_shipment ms on sl.shipment_id = ms.shipment_id and sl.log_timestamp = ms.latest_time
;

-- approach 2 : with window function
SELECT shipment_id , status_text as  current_status , log_timestamp as  last_updated
FROM shipment_logs
where row_number() over(partition by shipment_id order by  log_timestamp desc) = 1

In [ ]:
-- exclude books available less than 1 month
-- return books that sold less than 10 copy in the last year
with selected_books as (
    SELECT book_id , name
    FROM books
    where available_from < date_add('2019-06-23', interval -1 month)
)
SELECT book_id
from orders  o
join selected_books b on o.book_id = b.book_id
where extract(year FROM dispatch_date) = 2018
